In [4]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import torch
import requests
import mygene
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments

f:\code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import scanpy as sc

# Read the h5ad file
adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Display basic information
print(adata)

# Show observation metadata (cells)
print("\nOBS (cell metadata):")
print(adata.obs.head())

# Show gene metadata
print("\nVAR (gene metadata):")
print(adata.var.head())

# Show expression matrix shape
print("\nShape of dataset:")
print(adata.shape)

# Display first few gene names
print("\nGene names:")
print(adata.var_names[:10])

# Display first few cell names
print("\nCell names:")
print(adata.obs_names[:10])

# Display expression matrix preview
print("\nExpression matrix:")
print(adata.X[:5])

AnnData object with n_obs × n_vars = 65899 × 5060
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name', 'split1', 'subgroup1', 'split2', 'subgroup2', 'split3', 'subgroup3', 'split4', 'subgroup4', 'split5', 'subgroup5', 'perturbation'
    var: 'gene_name'
    uns: 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pert2neighbor', 'rank_genes_groups_cov_all', 'top_non_dropout_de_20', 'top_non_zero_de_20'
    obsm: 'perturbation_neighbors', 'perturbation_neighbors_pca'

OBS (cell metadata):
                   condition cell_type dose_val  control  \
cell_barcode                                               
AAACATACACCGAT-1  CREB1+ctrl   K562(?)      1+1        0   
AAACATACAGAGAT-1        ctrl   K562(?)        1        1   
AAACATACCAGAAA-1        ctrl   K562(?)        1        1   
AAACATACGTTGAC-1        ctrl   K562(?)        1        1   
AAACATACTGTTCT-1        ctrl   K562(?)        1        1   

                          condition_name split1      subgroup1 split

In [7]:
import scanpy as sc

# Load dataset
adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Show gene metadata columns
print(adata.var.columns)

# Display first rows
print(adata.var.head())

Index(['gene_name'], dtype='object')
                    gene_name
ENSG00000228463    AP006222.2
ENSG00000272438  RP11-54O7.16
ENSG00000230699   RP11-54O7.1
ENSG00000223764   RP11-54O7.3
ENSG00000187634        SAMD11


In [10]:
import scanpy as sc

adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Show first 20 genes
print(adata.var['gene_name'].head(20))

ENSG00000228463       AP006222.2
ENSG00000272438     RP11-54O7.16
ENSG00000230699      RP11-54O7.1
ENSG00000223764      RP11-54O7.3
ENSG00000187634           SAMD11
ENSG00000188290             HES4
ENSG00000187608            ISG15
ENSG00000162571           TTLL10
ENSG00000186891         TNFRSF18
ENSG00000186827          TNFRSF4
ENSG00000078808             SDF4
ENSG00000221978            CCNL2
ENSG00000272455    RP4-758J18.13
ENSG00000215915           ATAD3C
ENSG00000272106     RP11-345P4.9
ENSG00000197530             MIB2
ENSG00000189409           MMP23B
ENSG00000169885           CALML6
ENSG00000271806      RP5-892K4.1
ENSG00000182873    RP11-181G12.2
Name: gene_name, dtype: object


In [11]:
print(adata.var_names)

Index(['ENSG00000228463', 'ENSG00000272438', 'ENSG00000230699',
       'ENSG00000223764', 'ENSG00000187634', 'ENSG00000188290',
       'ENSG00000187608', 'ENSG00000162571', 'ENSG00000186891',
       'ENSG00000186827',
       ...
       'ENSG00000160255', 'ENSG00000227039', 'ENSG00000223768',
       'ENSG00000228355', 'ENSG00000182871', 'ENSG00000233922',
       'ENSG00000142156', 'ENSG00000228137', 'ENSG00000160298',
       'ENSG00000160299'],
      dtype='object', length=5060)


In [12]:
!pip install mygene -q

In [13]:
import scanpy as sc
import mygene
import pandas as pd

# Load dataset
adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Get gene names
genes = adata.var['gene_name'].tolist()

# Initialize mygene
mg = mygene.MyGeneInfo()

# Query gene information
result = mg.querymany(
    genes,
    scopes='symbol',
    fields='symbol,name,type_of_gene',
    species='human'
)

# Convert to DataFrame
df = pd.DataFrame(result)

# Display
print(df[['query', 'symbol', 'type_of_gene']].head(20))

49 input query terms found dup hits:	[('NFYC-AS1', 2), ('CYP4A22-AS1', 2), ('GAS5-AS1', 2), ('RPS10P7', 2), ('TRAPPC12-AS1', 2), ('UBE2E1
1339 input query terms found no hit:	['AP006222.2', 'RP11-54O7.16', 'RP11-54O7.1', 'RP11-54O7.3', 'RP4-758J18.13', 'RP11-345P4.9', 'RP5-8


            query    symbol    type_of_gene
0      AP006222.2       NaN             NaN
1    RP11-54O7.16       NaN             NaN
2     RP11-54O7.1       NaN             NaN
3     RP11-54O7.3       NaN             NaN
4          SAMD11    SAMD11  protein-coding
5            HES4      HES4  protein-coding
6           ISG15     ISG15  protein-coding
7          TTLL10    TTLL10  protein-coding
8        TNFRSF18  TNFRSF18  protein-coding
9         TNFRSF4   TNFRSF4  protein-coding
10           SDF4      SDF4  protein-coding
11          CCNL2     CCNL2  protein-coding
12  RP4-758J18.13       NaN             NaN
13         ATAD3C    ATAD3C  protein-coding
14   RP11-345P4.9       NaN             NaN
15           MIB2      MIB2  protein-coding
16         MMP23B    MMP23B  protein-coding
17         CALML6    CALML6  protein-coding
18    RP5-892K4.1       NaN             NaN
19  RP11-181G12.2       NaN             NaN


In [14]:
import scanpy as sc
import mygene
import pandas as pd

# Load dataset
adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Get gene names
genes = adata.var['gene_name'].tolist()

print("Total genes:", len(genes))

# Check repeated genes
duplicates = pd.Series(genes).value_counts()
repeated = duplicates[duplicates > 1]

print("\nRepeated genes:")
print(repeated)

print("\nNumber of repeated gene names:", len(repeated))

# Query gene types from MyGene
mg = mygene.MyGeneInfo()

result = mg.querymany(
    genes,
    scopes='symbol',
    fields='symbol,type_of_gene',
    species='human',
    as_dataframe=True
)

# Remove rows without gene type
result = result.dropna(subset=['type_of_gene'])

# Count gene types
gene_type_counts = result['type_of_gene'].value_counts()

print("\nGene Type Counts:")
print(gene_type_counts)

Total genes: 5060

Repeated genes:
Series([], Name: count, dtype: int64)

Number of repeated gene names: 0


49 input query terms found dup hits:	[('NFYC-AS1', 2), ('CYP4A22-AS1', 2), ('GAS5-AS1', 2), ('RPS10P7', 2), ('TRAPPC12-AS1', 2), ('UBE2E1
1339 input query terms found no hit:	['AP006222.2', 'RP11-54O7.16', 'RP11-54O7.1', 'RP11-54O7.3', 'RP4-758J18.13', 'RP11-345P4.9', 'RP5-8



Gene Type Counts:
type_of_gene
protein-coding    3530
ncRNA              183
pseudo               4
unknown              3
scRNA                1
Name: count, dtype: int64


In [1]:
!pip install mygene requests pandas tqdm

In [7]:
import scanpy as sc
import pandas as pd
import mygene
from collections import Counter

# -----------------------------
# STEP 3: LOAD ADAMSON DATASET
# -----------------------------
adamson_path = "adamson_biolord.h5ad"   # Change if needed

adata = sc.read_h5ad(adamson_path)

# -----------------------------
# STEP 4: EXTRACT GENE LIST
# -----------------------------
genes = list(adata.var_names)

print("======================================================")
print("TOTAL GENES IN ADAMSON DATASET:", len(genes))
print("======================================================")

# -----------------------------
# STEP 5: QUERY MYGENE DATABASE
# -----------------------------
mg = mygene.MyGeneInfo()

results = mg.querymany(
    genes,
    scopes="ensembl.gene", # Changed from "symbol" to "ensembl.gene"
    fields="symbol,name,entrezgene,ensembl.gene,type_of_gene,uniprot",
    species="human",
    as_dataframe=True,
    returnall=False,
    verbose=False
)

# -----------------------------
# STEP 6: CLEAN DATA
# -----------------------------
results = results.reset_index().rename(columns={"query": "Input_Gene"})

def check_yes_no(x):
    if pd.isna(x):
        return "No"
    # If it's a list or dict, check if it's empty
    if isinstance(x, (list, dict, set, tuple)):
        if not x: # empty list/dict/set/tuple means no info
            return "No"
    return "Yes"

# Add a check for 'entrezgene' column existence
if "entrezgene" in results.columns:
    results["NCBI"] = results["entrezgene"].apply(check_yes_no)
else:
    results["NCBI"] = "No" # If column doesn't exist, assume no NCBI info

# Fix for 'ensembl.gene' column - using the correct field name and robust check
if "ensembl.gene" in results.columns: # Changed from "ensembl" to "ensembl.gene"
    results["Ensembl"] = results["ensembl.gene"].apply(check_yes_no) # Changed from "ensembl" to "ensembl.gene"
else:
    results["Ensembl"] = "No"

if "uniprot" in results.columns:
    results["UniProt"] = results["uniprot"].apply(check_yes_no)
else:
    results["UniProt"] = "No"

# Add a check for 'type_of_gene' column existence
if "type_of_gene" in results.columns:
    results["type_of_gene"] = results["type_of_gene"].fillna("unknown")
else:
    results["type_of_gene"] = "unknown" # If column doesn't exist, create it and fill with "unknown"

# -----------------------------
# STEP 7: SUMMARY
# -----------------------------
total_genes = len(results)
ncbi_yes = (results["NCBI"] == "Yes").sum()
ensembl_yes = (results["Ensembl"] == "Yes").sum()
uniprot_yes = (results["UniProt"] == "Yes").sum()

print("\n================ DATABASE AVAILABILITY SUMMARY ================\n")
print(f"Total Genes Checked        : {total_genes}")
print(f"Available in NCBI          : {ncbi_yes}")
print(f"Available in Ensembl       : {ensembl_yes}")
print(f"Available in UniProt       : {uniprot_yes}")
print(f"Missing in NCBI            : {total_genes - ncbi_yes}")
print(f"Missing in Ensembl         : {total_genes - ensembl_yes}")
print(f"Missing in UniProt         : {total_genes - uniprot_yes}")

# -----------------------------
# STEP 8: GENE TYPE COUNTS
# -----------------------------
print("\n================ GENE TYPE COUNTS ================\n")

gene_type_counts = results["type_of_gene"].value_counts()

for gene_type, count in gene_type_counts.items():
    print(f"{gene_type:25} : {count}")

# -----------------------------
# STEP 9: SHOW GENES BY TYPE
# -----------------------------
print("\n================ GENES GROUPED BY TYPE ================\n")

for gene_type in gene_type_counts.index:
    gene_subset = results[results["type_of_gene"] == gene_type]["Input_Gene"].tolist()

    print(f"\n--- {gene_type.upper()} ({len(gene_subset)} genes) ---")

    # Print first 50 to avoid huge output
    for g in gene_subset[:50]:
        print(g)

    if len(gene_subset) > 50:
        print(f"... and {len(gene_subset)-50} more")

# -----------------------------
# STEP 10: MISSING GENES
# -----------------------------
print("\n================ MISSING IN NCBI ================\n")
missing_ncbi = results[results["NCBI"] == "No"]["Input_Gene"].tolist()

if len(missing_ncbi) == 0:
    print("No missing genes in NCBI")
else:
    for g in missing_ncbi[:100]:
        print(g)
    if len(missing_ncbi) > 100:
        print(f"... and {len(missing_ncbi)-100} more")

print("\n================ MISSING IN ENSEMBL ================\n")
missing_ensembl = results[results["Ensembl"] == "No"]["Input_Gene"].tolist()

if len(missing_ensembl) == 0:
    print("No missing genes in Ensembl")
else:
    for g in missing_ensembl[:100]:
        print(g)
    if len(missing_ensembl) > 100:
        print(f"... and {len(missing_ensembl)-100} more")

print("\n================ MISSING IN UNIPROT ================\n")
missing_uniprot = results[results["UniProt"] == "No"]["Input_Gene"].tolist()

if len(missing_uniprot) == 0:
    print("No missing genes in UniProt")
else:
    for g in missing_uniprot[:100]:
        print(g)
    if len(missing_uniprot) > 100:
        print(f"... and {len(missing_uniprot)-100} more")

# -----------------------------
# STEP 11: FULL TABLE PREVIEW
# -----------------------------
print("\n================ FULL GENE DATABASE TABLE (TOP 100) ================\n")

display(
    results[
        ["Input_Gene", "symbol", "name", "type_of_gene", "NCBI", "Ensembl", "UniProt"]
    ].head(100)
)

# -----------------------------
# STEP 12: FINAL COUNTS
# -----------------------------
print("\n======================================================")
print("FINAL ANALYSIS COMPLETE")
print("You now know:")
print("1. Total genes in Adamson")
print("2. Gene types")
print("3. NCBI availability")
print("4. Ensembl availability")
print("5. UniProt availability")
print("6. Missing genes")
print("======================================================")

TOTAL GENES IN ADAMSON DATASET: 5060

================ DATABASE AVAILABILITY SUMMARY ================

Total Genes Checked        : 5061
Available in NCBI          : 4214
Available in Ensembl       : 4552
Available in UniProt       : 0
Missing in NCBI            : 847
Missing in Ensembl         : 509
Missing in UniProt         : 5061

================ GENE TYPE COUNTS ================

protein-coding            : 3753
unknown                   : 847
ncRNA                     : 453
pseudo                    : 8

================ GENES GROUPED BY TYPE ================


--- PROTEIN-CODING (3753 genes) ---
ENSG00000187634
ENSG00000188290
ENSG00000187608
ENSG00000162571
ENSG00000186891
ENSG00000186827
ENSG00000078808
ENSG00000221978
ENSG00000215915
ENSG00000197530
ENSG00000189409
ENSG00000169885
ENSG00000157873
ENSG00000235169
ENSG00000187017
ENSG00000049249
ENSG00000116285
ENSG00000142599
ENSG00000074800
ENSG00000049239
ENSG00000178585
ENSG00000054523
ENSG00000132879
ENSG00000177000
ENSG0

,Input_Gene,symbol,name,type_of_gene,NCBI,Ensembl,UniProt
0,ENSG00000228463,RPL23AP21,ribosomal protein L23a pseudogene 21,pseudo,Yes,Yes,No
1,ENSG00000272438,NaN,NaN,unknown,No,Yes,No
2,ENSG00000230699,NaN,NaN,unknown,No,No,No
3,ENSG00000223764,LINC02593,long intergenic non-protein coding RNA 2593,ncRNA,Yes,Yes,No
4,ENSG00000187634,SAMD11,sterile alpha motif domain containing 11,protein-coding,Yes,Yes,No
...,...,...,...,...,...,...,...
95,ENSG00000229388,TAF12-DT,TAF12 divergent transcript,unknown,No,Yes,No
96,ENSG00000162419,GMEB1,glucocorticoid modulatory element binding prot...,protein-coding,Yes,Yes,No
97,ENSG00000198492,YTHDF2,YTH N6-methyladenosine RNA binding protein F2,protein-coding,Yes,Yes,No
98,ENSG00000159023,EPB41,erythrocyte membrane protein band 4.1,protein-coding,Yes,Yes,No



FINAL ANALYSIS COMPLETE
You now know:
1. Total genes in Adamson
2. Gene types
3. NCBI availability
4. Ensembl availability
5. UniProt availability
6. Missing genes


In [4]:
# =========================================================
# SIMPLE GENE AVAILABILITY CHECK (AVAILABLE / NOT AVAILABLE)
# Checks MyGene + NCBI-level symbol recognition
# =========================================================

import scanpy as sc
import mygene
import pandas as pd

# Load dataset
adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Get genes
if 'gene_name' in adata.var.columns:
    genes = adata.var['gene_name'].astype(str).tolist()
else:
    genes = adata.var_names.astype(str).tolist()

print("Total genes:", len(genes))

# Initialize MyGene
mg = mygene.MyGeneInfo()

# Batch query
result = mg.querymany(
    genes,
    scopes='symbol',
    fields='symbol',
    species='human'
)

# =========================================================
# MARK AVAILABLE / NOT AVAILABLE
# =========================================================
availability = []

for r in result:
    gene = r.get("query")

    if r.get("notfound", False):
        status = "NOT_AVAILABLE"
    else:
        status = "AVAILABLE"

    availability.append({
        "gene_name": gene,
        "availability": status
    })

# Convert to DataFrame
availability_df = pd.DataFrame(availability)

# =========================================================
# SUMMARY
# =========================================================
print("\nGene Availability Summary:")
print(availability_df["availability"].value_counts())

# =========================================================
# SAVE FILES
# =========================================================
availability_df.to_csv(r"F:\code\gene_availability.csv", index=False)

# Missing only
not_available_df = availability_df[
    availability_df["availability"] == "NOT_AVAILABLE"
]

not_available_df.to_csv(
    r"F:\code\not_available_genes.csv",
    index=False
)

# =========================================================
# DISPLAY
# =========================================================
print("\nTop 20 Results:")
print(availability_df.head(20))

print("\nFiles Saved:")
print("1. F:\\code\\gene_availability.csv")
print("2. F:\\code\\not_available_genes.csv")

Total genes: 5060


49 input query terms found dup hits:	[('NFYC-AS1', 2), ('CYP4A22-AS1', 2), ('GAS5-AS1', 2), ('RPS10P7', 2), ('TRAPPC12-AS1', 2), ('UBE2E1
1339 input query terms found no hit:	['AP006222.2', 'RP11-54O7.16', 'RP11-54O7.1', 'RP11-54O7.3', 'RP4-758J18.13', 'RP11-345P4.9', 'RP5-8



Gene Availability Summary:
availability
AVAILABLE        3788
NOT_AVAILABLE    1339
Name: count, dtype: int64

Top 20 Results:
        gene_name   availability
0      AP006222.2  NOT_AVAILABLE
1    RP11-54O7.16  NOT_AVAILABLE
2     RP11-54O7.1  NOT_AVAILABLE
3     RP11-54O7.3  NOT_AVAILABLE
4          SAMD11      AVAILABLE
5            HES4      AVAILABLE
6           ISG15      AVAILABLE
7          TTLL10      AVAILABLE
8        TNFRSF18      AVAILABLE
9         TNFRSF4      AVAILABLE
10           SDF4      AVAILABLE
11          CCNL2      AVAILABLE
12  RP4-758J18.13  NOT_AVAILABLE
13         ATAD3C      AVAILABLE
14   RP11-345P4.9  NOT_AVAILABLE
15           MIB2      AVAILABLE
16         MMP23B      AVAILABLE
17         CALML6      AVAILABLE
18    RP5-892K4.1  NOT_AVAILABLE
19  RP11-181G12.2  NOT_AVAILABLE

Files Saved:
1. F:\code\gene_availability.csv
2. F:\code\not_available_genes.csv


In [5]:
# =========================================================
# GENE DATABASE SOURCE CHECK
# Output:
# AVAILABLE_IN_NCBI
# AVAILABLE_IN_UNIPROT
# AVAILABLE_IN_BOTH
# NOT_AVAILABLE
# =========================================================

import scanpy as sc
import pandas as pd
import requests
from tqdm import tqdm
import time

# =========================================================
# LOAD DATASET
# =========================================================
adata = sc.read_h5ad(r"F:\code\adamson_biolord.h5ad")

# Get genes
if 'gene_name' in adata.var.columns:
    genes = adata.var['gene_name'].astype(str).tolist()
else:
    genes = adata.var_names.astype(str).tolist()

print("Total genes:", len(genes))


# =========================================================
# NCBI CHECK
# =========================================================
def check_ncbi(gene):
    try:
        url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=gene&term={gene}[sym]+AND+human[orgn]&retmode=json"
        r = requests.get(url, timeout=10)

        if r.status_code == 200:
            ids = r.json()["esearchresult"]["idlist"]
            return len(ids) > 0
        return False

    except:
        return False


# =========================================================
# UNIPROT CHECK
# =========================================================
def check_uniprot(gene):
    try:
        url = f"https://rest.uniprot.org/uniprotkb/search?query=gene:{gene}+AND+organism_id:9606&format=json&size=1"
        r = requests.get(url, timeout=10)

        if r.status_code == 200:
            data = r.json()

            if "results" in data:
                return len(data["results"]) > 0

        return False

    except:
        return False


# =========================================================
# MAIN LOOP
# =========================================================
results = []

for gene in tqdm(genes):

    ncbi_found = check_ncbi(gene)
    uniprot_found = check_uniprot(gene)

    # Determine source
    if ncbi_found and uniprot_found:
        status = "AVAILABLE_IN_BOTH"
    elif ncbi_found:
        status = "AVAILABLE_IN_NCBI"
    elif uniprot_found:
        status = "AVAILABLE_IN_UNIPROT"
    else:
        status = "NOT_AVAILABLE"

    results.append({
        "gene_name": gene,
        "database_status": status
    })

    # Avoid API overload
    time.sleep(0.2)


# =========================================================
# DATAFRAME
# =========================================================
db_df = pd.DataFrame(results)


# =========================================================
# SUMMARY
# =========================================================
print("\nDatabase Availability Summary:")
print(db_df["database_status"].value_counts())


# =========================================================
# SAVE FILE
# =========================================================
db_df.to_csv(r"F:\code\gene_database_source.csv", index=False)


# Separate files
db_df[db_df["database_status"] == "AVAILABLE_IN_BOTH"].to_csv(
    r"F:\code\genes_in_both.csv", index=False
)

db_df[db_df["database_status"] == "AVAILABLE_IN_NCBI"].to_csv(
    r"F:\code\genes_only_ncbi.csv", index=False
)

db_df[db_df["database_status"] == "AVAILABLE_IN_UNIPROT"].to_csv(
    r"F:\code\genes_only_uniprot.csv", index=False
)

db_df[db_df["database_status"] == "NOT_AVAILABLE"].to_csv(
    r"F:\code\genes_not_available.csv", index=False
)


# =========================================================
# DISPLAY
# =========================================================
print("\nTop 20 Results:")
print(db_df.head(20))

print("\nSaved Files:")
print("1. gene_database_source.csv")
print("2. genes_in_both.csv")
print("3. genes_only_ncbi.csv")
print("4. genes_only_uniprot.csv")
print("5. genes_not_available.csv")

Total genes: 5060


100%|██████████| 5060/5060 [1:21:26<00:00,  1.04it/s]  


Database Availability Summary:
database_status
NOT_AVAILABLE        3932
AVAILABLE_IN_BOTH    1065
AVAILABLE_IN_NCBI      63
Name: count, dtype: int64

Top 20 Results:
        gene_name    database_status
0      AP006222.2      NOT_AVAILABLE
1    RP11-54O7.16      NOT_AVAILABLE
2     RP11-54O7.1      NOT_AVAILABLE
3     RP11-54O7.3      NOT_AVAILABLE
4          SAMD11  AVAILABLE_IN_BOTH
5            HES4  AVAILABLE_IN_BOTH
6           ISG15  AVAILABLE_IN_BOTH
7          TTLL10  AVAILABLE_IN_BOTH
8        TNFRSF18  AVAILABLE_IN_BOTH
9         TNFRSF4  AVAILABLE_IN_BOTH
10           SDF4  AVAILABLE_IN_BOTH
11          CCNL2  AVAILABLE_IN_BOTH
12  RP4-758J18.13      NOT_AVAILABLE
13         ATAD3C  AVAILABLE_IN_BOTH
14   RP11-345P4.9      NOT_AVAILABLE
15           MIB2  AVAILABLE_IN_BOTH
16         MMP23B  AVAILABLE_IN_BOTH
17         CALML6  AVAILABLE_IN_BOTH
18    RP5-892K4.1      NOT_AVAILABLE
19  RP11-181G12.2      NOT_AVAILABLE

Saved Files:
1. gene_database_source.csv
2. genes_in_